# Install Package

In [1]:
# 1. Nâng cấp công cụ lõi
!pip install -U pip "setuptools<82.0.0" "jedi>=0.16"

# 2. Cài đặt hệ sinh thái Unsloth & Xformers (Tự động kéo Torch bản mới nhất)
!pip install unsloth==2026.5.2 unsloth_zoo xformers

# 3. Cài đặt TRL (mã nguồn mới nhất) và các vệ tinh cho GRPO
!pip install git+https://github.com/huggingface/trl.git@main --no-deps
!pip install llm-blender mergekit verlib

  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 1.5.0.dev0
    Uninstalling trl-1.5.0.dev0:
      Successfully uninstalled trl-1.5.0.dev0
  Cloning https://github.com/huggingface/trl.git (to revision main) to /tmp/pip-req-build-1p3t6z88
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/trl.git /tmp/pip-req-build-1p3t6z88
  Resolved https://github.com/huggingface/trl.git to commit 16f6d959ab50961ff2ab0860a5452b1b877b9775
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for trl: filename=trl-1.5.0.dev0-py3-none-any.whl size=752655 sha256=d185bbf10319109e49a3d15aba885977970f6f5f5ff219419ca5e293b70f18ab
  Stored in directory: /tmp/pip-ephem-wheel-cache-at44na45/wheels/e7/c7/c7/0362f725a459257a13c6f4d02914a865acb5ba2a50cbd1e5e0
Su

# Import Library

In [2]:
import os
import re
import torch
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel

/tmp/ipykernel_18644/3491554547.py:5: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Load BaseModel

In [3]:
# Bật tính năng Standby của Unsloth để tiết kiệm 30%+ bộ nhớ cho RL
os.environ['UNSLOTH_VLLM_STANDBY'] = "1"

max_seq_length = 2048 # Có thể tăng lên nếu suy luận dài
lora_rank = 32 # Rank cao hơn = thông minh hơn, nhưng train chậm hơn

# Tải mô hình cơ sở
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B", # Tên model LLM
    max_seq_length = max_seq_length,  # độ dài của context length mà mô hình có thể đọc và sinh ra trong 1 chu kỳ
    load_in_4bit = True,               # quantization xuống 4-bit
    fast_inference = False,           # tắt suy luận nhanh để ổn định hệ thống
    max_lora_rank = lora_rank,        # rank của LoRA
    load_in_fp8 = False,               # tắt chế độ tải mô hình bằng định dạng độ chính xác Float8
)
# Cấu hình LoRA Adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # chọn module của model để áp dụng LoRA
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",  # dọn dẹp bộ nhớ rác sinh ra khi train giảm VRAM
    random_state = 3407,
)

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


unsloth/deepseek-r1-0528-qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


# Chat template

In [4]:
reasoning_start = "<think>\n"
reasoning_end = "</think>\n"
solution_start = "<answer>\n"
solution_end = "\n</answer>"

system_prompt = f"""You are given a problem. Think about the problem and provide your working out. Place it between {reasoning_start} and {reasoning_end}. Then, provide your solution between {solution_start}{solution_end}""" # [4]

tokenizer.chat_template = tokenizer.chat_template \
    .replace("'{system_prompt}'", f"'{system_prompt}'") \
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")


# Dataset

In [5]:
from datasets import load_dataset
print("2. Đang tải dataset Open R1 Math...")
# Lấy trực tiếp dataset dùng trong template của Unsloth
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train") # [1]

def format_grpo_data(x):
    return {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": x["prompt"]}
        ],
        "answer": x["solution"] # [5]
    }

# Lấy 500 mẫu đầu tiên để chạy thử cho nhanh trên Colab
train_dataset = dataset.select(range(500)).map(format_grpo_data)


2. Đang tải dataset Open R1 Math...


README.md: 0.00B [00:00, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/5.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14116 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

# REWARD FUNCTIONS



In [ ]:
import re

match_numbers = re.compile(r"-?\d+\.?\d*")

import re

match_numbers = re.compile(r"-?\d+\.?\d*")

import re

match_numbers = re.compile(r"-?\d+\.?\d*")

def reward_fn(completions, **kwargs):
    scores = []

    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else str(c)

        score = 0.0

        # FORMAT (nhẹ hơn)
        if text.count("<think>") == 1 and text.count("</think>") == 1:
            score += 1.5
        else:
            score -= 0.5

        # reasoning
        think_part = text.split("<think>")[-1]
        score += min(len(think_part.split()) / 200, 1.0)

        # numbers (anti-spam)
        nums = match_numbers.findall(text)
        score += min(len(nums[:3]) * 0.1, 0.3)

        # spam penalty
        if len(nums) > 15:
            score -= 0.5

        scores.append(score)

    print("AVG:", sum(scores)/len(scores))
    return scores

# Config

In [7]:
# !pip install vllm

In [27]:
# from vllm import SamplingParams
# vllm_sampling_params = SamplingParams(
#     min_p = 0.1,                        # bỏ những từ có xs xuất hiện thấp hơn 10% so với từ có xs cao nhất
#     top_p = 1.0, top_k = -1,            # đặt vậy cơ bản để tắt 2 arg này, nhường sự sáng tạo  cho min_p
#     seed = 3407,
#     stop = [tokenizer.eos_token],       # ký tự kết thúc câu (end of sequence)
#     include_stop_str_in_output = True,  # giữ lại thẻ EOS đó trong chuỗi kết quả cuối cùng
# )

training_args = GRPOConfig(
    # vllm_sampling_params = vllm_sampling_params,            # Đã khai báo ở trên
    output_dir = "/content/drive/MyDrive/grpo_checkpoints", # Trỏ đường dẫn lưu thẳng vào Google Drive
    save_steps = 50,                                        # Cứ train được 50 bước thì tự động lưu 1 bản checkpoint
    save_total_limit = 3,                                   # Chỉ giữ lại 3 thư mục checkpoint mới nhất để tránh đầy dung lượng Drive
    temperature = 0.7,                                      # để sáng tạo 0.7 để mô hình thử nhiều cách lập luận
    learning_rate = 5e-6,
    weight_decay = 0.01,
    bf16=False,
    fp16=False,
    max_grad_norm=1.0,
    warmup_steps = 5,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",                                   # dùng optimaztion 8bit giảm bộ nhớ
    logging_steps = 1,
    per_device_train_batch_size = 2,                        # số lượng câu input
    gradient_accumulation_steps = 2,                        # cộng dồn gradient
    num_generations = 2,                                    # gen ra 2 câu trả lời cho cùng 1 bài toán sau đó so sánh điểm để chọn câu điều chỉnh weight model
    max_completion_length = 512,                            # max của câu trả lời
    max_steps = 50,
    report_to = "none",
    data_seed = 3407
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs=[reward_fn],
    args = training_args,
    train_dataset = train_dataset,
)

In [28]:
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

In [30]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 87,293,952 of 8,278,029,312 (1.05% trained)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [ ]:
from unsloth import FastLanguageModel

# Giả sử 'model' và 'tokenizer' là cái ông đã load thành công ở bước [3]
# Thử convert sang GGUF bản 4-bit (phổ biến nhất)
model.save_pretrained_gguf(
    "test_model_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)